In [ ]:
import numpy as np
import os
import sys

sys.path.append('../../..')
path=os.path.abspath(os.path.join(os.getcwd(), "../../../outputs/linear"))

import torch
import torchsde
from torch import nn, profiler, Tensor, nan, vmap
import copy
from types import SimpleNamespace

from tqdm import tqdm
import contextlib
import io

from core.utils import  ema_update_trackers, update_ema_model, generate_system_canon, sample_linear, compute_gradnorm, save_model, loss_compute
from core.policies_lqr import  HyperLinearControl,  find_params
from configs.linear_cfg import cfg
from adabelief_pytorch import AdaBelief

is_cuda = torch.cuda.is_available()
device = 'cuda' if is_cuda else 'cpu'

if not is_cuda:
    print("Warning: CUDA not available; falling back to CPU but this is likely to be very slow.")
import gc


In [ ]:
ctx = {'device': device, 'dtype': torch.float32}

# Clean, one-liner conversion
At, Bt = [torch.as_tensor(x, **ctx) for x in generate_system_canon(cfg.dim)]
coremat=(At.unsqueeze(0)).repeat(cfg.batch_size,1,1)

In [ ]:
def train_lqr_model(layers, config):
    """
    Trains a single LQR model instance for a specific number of layers.
    """
    # 1. Initialization
    model = HyperLinearControl(
        At, Bt, config.batch_size, device, layers, 
        config.strategy, config.skip, config.activation, 
        config.clip_val, config.std_scales
    ).to(device)
        
    with contextlib.redirect_stdout(io.StringIO()): 
        
        ema_model = copy.deepcopy(model)
        optimizer = AdaBelief(find_params(model), lr=config.learning_rate, rectify=True)
        scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=config.gamma)
    
    # Tracking containers
    stats = {k: [] for k in ['eperf', 'stperf', 'eloss', 'emaperf', 'gradnorms']}
    ema_vals = [0.0] * 4 # [loss, ener, enr_std, std]
    
    ts_exp = torch.linspace(0, config.t_total, config.t_size).to(device)
    dt = ts_exp[1] - ts_exp[0]
    
    score=config.score
    # 2. Training Loop
    pbar = tqdm(range(config.iterations), desc=f"Layers: {layers}")
    for i in pbar:
        # Data Preparation
        amat, x0 = sample_linear(config.dim, config.batch_size, device)
        ys_exp = torch.cat((x0, torch.zeros(config.batch_size, 1).to(device)), dim=1)
        
        # Update system matrices in model
        coremat[:, -1, :] = amat
        model.At, model.poly.amat = coremat, amat

        # Forward Pass (SDE Integration)
        ys_tray = torchsde.sdeint_adjoint(
            model, ys_exp, ts_exp, method='reversible_heun', dt=dt,
            adjoint_method='adjoint_reversible_heun'
        )
        
        # Loss & Backprop
        loss_val, log_trick, std_v, energy, std_en = loss_compute(ys_tray, config.dim, config.weight)
        
        optimizer.zero_grad(set_to_none=True)
        log_trick.backward()
        grad_norm = compute_gradnorm(model)
        optimizer.step()
        
        # Post-step updates
        model.project(config.strategy, config.clip_val)
        
        scheduler.step()
        update_ema_model(model, ema_model, config.alpha)
        
        # Update EMA trackers
        ema_vals = ema_update_trackers(i, config.alpha, loss_val, energy, std_en, std_v, *ema_vals)
        
        # Log to Progress Bar 
        pbar.set_postfix({
            "Loss": f"{loss_val:.5f}",
            "EMA": f"{ema_vals[0]:.5f}",
            "Grad": f"{grad_norm:.1e}"
        })
        score= save_model(score, ema_vals[0], ema_model, model, path,config.strategy,layers)
     
        # Memory Management
        if i % 200 == 0:
            gc.collect()
            torch.cuda.empty_cache()

        if loss_val > 500:
            print(f"Early stopping at iter {i}: Loss diverged.")
            break
            
    return stats, ema_model

In [ ]:
# Main Execution Loop
results = []

for layer_count in range(1, 5):
    print(f"\n--- Starting Experiment: {layer_count} Layers ---")
    run_stats, best_model = train_lqr_model(layer_count, cfg)
    results.append(run_stats)
    
    # Optional: Clear output after each experiment to keep the notebook tidy
    # clear_output(wait=True)

print("All experiments completed.")